# stats-transformer: Config-Driven Econometrics in YAML

The point of `stats-transformer` is to move transformations and econometric specifications out of ad-hoc Python scripts and into auditable YAML files under `references/configs/`.

Below we run **three different econometric workflows** (log-transformed Robust OLS, two-way fixed-effects Panel OLS, and a macro VAR) using the **exact same two lines of Python** — only the YAML config changes.

In [1]:
import numpy as np
import statsmodels.api as sm
from stats_transformer import Pipeline
from stats_transformer.data import load_example
from economic_libraries.notebooks import setup_repo_root

REPO_ROOT = setup_repo_root()

## 1. Featurization + HC3 Robust OLS (YAML Pipeline)

Inspect the config: `references/configs/stats_transformer/longley_robust_ols.yaml` declares the log transforms (`GNP`, `POP`) and `robust_ols` (`HC3`) in one place.

In [2]:
cfg_ols = REPO_ROOT / "references/configs/stats_transformer/longley_robust_ols.yaml"
print(cfg_ols.read_text())

# YAML-driven featurization + HC3 Robust OLS on Longley macroeconomic data.
# Consumed by notebooks/1_stats_transformer.ipynb via stats_transformer.Pipeline.
data:
  featurization:
    date_column: date
    column_mapping:
      YEAR: date
    transformations:
      - log
    data_columns:
      - GNP
      - POP

model:
  model_type: robust_ols
  target_variable: TOTEMP
  independent_variables:
    - GNPDEFL
    - GNP_log
    - UNEMP
    - ARMED
    - POP_log
  robust_ols:
    cov_type: HC3



In [3]:
longley = sm.datasets.longley.load_pandas().data
pipe_ols = Pipeline(params_path=str(cfg_ols))
longley_feat = pipe_ols.fit_transform(longley)

print(pipe_ols.model.get_summary())

                            OLS Regression Results                            
Dep. Variable:                 TOTEMP   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.985
Method:                 Least Squares   F-statistic:                     99.17
Date:                Tue, 24 Nov 2026   Prob (F-statistic):           3.45e-08
Time:                        03:37:41   Log-Likelihood:                -115.71
No. Observations:                  16   AIC:                             243.4
Df Residuals:                      10   BIC:                             248.1
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const      -3.001e+05   2.98e+05     -1.006      0.3

### Parity Check Against Native `statsmodels`

Verify that the YAML pipeline produces the exact same coefficients and HC3 robust standard errors as hand-coded `statsmodels`.

In [4]:
X = sm.add_constant(longley_feat[["GNPDEFL", "GNP_log", "UNEMP", "ARMED", "POP_log"]])
sm_fit = sm.OLS(longley_feat["TOTEMP"], X).fit(cov_type="HC3")

assert np.allclose(sm_fit.params.sort_index().values, pipe_ols.model.model.params.sort_index().values)
assert np.allclose(sm_fit.bse.sort_index().values, pipe_ols.model.model.bse.sort_index().values)
print("YAML Robust OLS matches native statsmodels (coefficients + HC3 SEs).")

YAML Robust OLS matches native statsmodels (coefficients + HC3 SEs).


## 2. Swap Only the YAML: Two-Way Fixed-Effects Panel OLS (Grunfeld 1958)

Same `Pipeline(params_path=...).fit_transform(...)` call, pointed at `grunfeld_panel_pipeline.yaml`.

In [5]:
cfg_panel = REPO_ROOT / "references/configs/stats_transformer/grunfeld_panel_pipeline.yaml"
print(cfg_panel.read_text())

# YAML-driven two-way fixed-effects Panel OLS on Grunfeld (1958).
# Consumed by notebooks/1_stats_transformer.ipynb via stats_transformer.Pipeline.
data:
  featurization:
    entity_column: country
    date_column: date
    column_mapping:
      firm: country
      year: date

model:
  model_type: panel_ols
  target_variable: invest
  independent_variables:
    - value
    - capital
  panel_ols:
    entity_effects: true
    time_effects: true



In [6]:
pipe_panel = Pipeline(params_path=str(cfg_panel))
pipe_panel.fit_transform(load_example("grunfeld"))

print(pipe_panel.model.get_summary())

                          PanelOLS Estimation Summary                           
Dep. Variable:                 invest   R-squared:                        0.7253
Estimator:                   PanelOLS   R-squared (Between):              0.7944
No. Observations:                 220   R-squared (Within):               0.7566
Date:                Tue, Nov 24 2026   R-squared (Overall):              0.7856
Time:                        03:37:41   Log-likelihood                   -1153.0
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      248.15
Entities:                          11   P-value                           0.0000
Avg Obs:                       20.000   Distribution:                   F(2,188)
Min Obs:                       20.000                                           
Max Obs:                       20.000   F-statistic (robust):             248.15
                            

/tmp/venv-nb-test/lib/python3.12/site-packages/stats_transformer/featurization/feature_engineering.py:382: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  transformed_df = grouped.apply(self._apply_entity_transformations).reset_index(drop=True)


## 3. Swap Only the YAML Again: Reduced-Form Macro VAR

Point the exact same two lines at `var.yaml` (US real GDP, consumption, investment, `maxlags: 2`).

In [7]:
cfg_var = REPO_ROOT / "references/configs/stats_transformer/var.yaml"
print(cfg_var.read_text())

# Reduced-form VAR on US real GDP, consumption and investment.
# Consumed by src/economic_libraries/stats_transformer/var.py
# The data ship inside stats-transformer; see stats_transformer.data.load_example.
data:
  example_dataset: macrodata

model:
  model_type: var
  target_variables: [realgdp, realcons, realinv]
  maxlags: 2



In [8]:
pipe_var = Pipeline(params_path=str(cfg_var))
pipe_var.fit_transform(load_example("macrodata"))

print(pipe_var.model.get_summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 24, Nov, 2026
Time:                     03:37:41
--------------------------------------------------------------------
No. of Equations:         3.00000    BIC:                  0.0480050
Nobs:                     200.000    HQIC:                 -0.158167
Log likelihood:          -800.531    FPE:                   0.742129
AIC:                    -0.298318    Det(Omega_mle):        0.669358
--------------------------------------------------------------------
Results for equation realgdp
                 coefficient       std. error           t-stat            prob
------------------------------------------------------------------------------
const               0.152697         0.111902            1.365           0.172
L1.realgdp         -0.279435         0.169663           -1.647           0.100
L1.realcons         0.675016         0.131285            5.142    

## CLI Equivalent

For full file-to-report runs (`data.raw_data_file` + `reporting.output_dir` in YAML), run straight from the terminal with zero Python:

```bash
uv run python -m stats_transformer.pipeline --config references/configs/<your_config>.yaml
```